# Signal Hunters R2 - run 10 (teammate A): FIXED normalisation - EfficientNet-B2 + B0, different seeds from run 7
GPU T4 x2, Persistence: Files only, Internet ON, input: signal-hunters-r2.

In [ ]:
%%writefile /kaggle/working/cnn_r2.py
"""Signal Hunters R2 - CNN on the 'global-time canvas'.
Each cadence -> one 128x128 image: the 6 panels placed on ONE time axis including the slew gaps
(panel p row r at t = p*(16+gap)+r), so a true ETI drift is a single straight line through the 3 ON panels.
Input channels: z (bandpass-flattened, noise-normalised), zr (z minus per-row median: kills broadband rows),
on/off indicator (+1 ON rows, -1 OFF rows, 0 gaps), valid mask.
Domain shift GBT->PKS is attacked with augmentation: frequency zoom (PKS drifts look ~0.64x as steep),
SNR loss, dropped rows, broadband glitch rows, per-panel gain, frequency shift/flip.
Usage: python cnn_r2.py DATA_DIR EPOCHS SEEDS [arch=res|timm_resnet18|timm_effb0] [PL]
Writes oof_<tag>.npy (18000x9) and test_<tag>.npy (10000x9) to ./out/"""
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fn, sys, os, time, math
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
LABELS = ['eti_candidate', 'frb', 'pulsar', 'scintillating_star', 'satellite_pass', 'radar_chirp',
          'rfi_intermittent', 'rfi_persistent', 'noise']
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
UP = int(os.environ.get('UP', 1)); P_ETI = float(os.environ.get('P_ETI', 0)); P_TRAP = float(os.environ.get('P_TRAP', 0))

def normalise(X):
    x = X.astype(np.float32); m = (X > 0); x[~m] = np.nan
    x = x - np.nanmedian(x, axis=2, keepdims=True)
    s = 1.4826 * np.nanmedian(np.abs(x - np.nanmedian(x, axis=(2, 3), keepdims=True)), axis=(2, 3), keepdims=True) + 1e-3
    z = np.nan_to_num(np.clip(x / s, -10, 60))
    return z.astype(np.float16), m

def load(D, split):
    X = np.load(f'{D}/{split}_cadences.npy'); meta = pd.read_csv(f'{D}/{split}_meta.csv')
    return X, (X > 0), meta.slew_gap_rows.values.astype(np.int64), meta

def norm_t(x, m):
    """FIXED normalisation (v2). Old version subtracted each channel's median over time, which ERASED every
    non-drifting line (persistent RFI, non-drifting traps): a 10-sigma persistent line became 0.4 sigma.
    Now: bandpass = per-panel time-median spectrum smoothed ACROSS FREQUENCY (median, 9 channels), so narrow
    lines survive. Channel 64 (GBT-only centre spike, absent in PKS) is blanked in every cadence."""
    x = torch.where(m > 0, x, torch.full_like(x, float('nan')))
    x[..., 64] = float('nan')
    spec = torch.nanmedian(x, dim=2).values                                   # (B,6,128)
    spec = torch.where(torch.isnan(spec), torch.nanmedian(spec, dim=-1, keepdim=True).values, spec)
    pad = Fn.pad(spec, (4, 4), mode='replicate')
    band = pad.unfold(-1, 9, 1).median(-1).values                            # smooth bandpass
    r = x - band[:, :, None, :]
    f = r.flatten(2); med = torch.nanmedian(f, dim=2, keepdim=True).values
    s = 1.4826 * torch.nanmedian((f - med).abs(), dim=2, keepdim=True).values + 1e-3
    z = ((r - med[..., None]) / s[..., None]).clamp(-10, 60)
    return torch.nan_to_num(z, nan=0.0)

def byte_aug(x, m, k_lo=5., k_hi=30., p=0.8):
    """PKS-like receiver noise in the POWER domain: power *= Gamma(k)/k  -> skewed log-noise like the test
    telescope (measured: noise MAD 15.6 vs 11.9 bytes, more negative skew). Also extra dropped/glitch rows."""
    B = x.shape[0]
    k = torch.empty(B, 1, 1, 1, device=x.device).uniform_(k_lo, k_hi)
    g = torch.distributions.Gamma(k.expand_as(x), k.expand_as(x)).sample().clamp_min(1e-4)
    sel = (torch.rand(B, 1, 1, 1, device=x.device) < p).float()
    x = x + sel * 24 * torch.log2(g)
    drop = (torch.rand(B, 6, 16, 1, device=x.device) < torch.empty(B, 1, 1, 1, device=x.device).uniform_(0, 0.03)).float()
    gl = (torch.rand(B, 6, 16, 1, device=x.device) < 0.01).float() * torch.empty(B, 6, 16, 1, device=x.device).uniform_(10, 50)
    x = (x + gl).clamp(1, 255)
    m = m * (1 - drop)
    return x, m

# ---------------- GPU-side augmentation + canvas -----------------
def augment(z, m, train):
    """z,m: (B,6,16,128) float on device"""
    B = z.shape[0]
    if not train: return z, m
    # SNR loss + extra noise (PKS noisier): z' = a*z + sqrt(1-a^2)*N
    a = torch.empty(B, 1, 1, 1, device=z.device).uniform_(0.75, 1.0)
    z = a * z + torch.sqrt(1 - a ** 2) * torch.randn_like(z)
    # per-panel gain on the signal part
    z = z * torch.empty(B, 6, 1, 1, device=z.device).uniform_(0.8, 1.2)
    # frequency zoom (compress/expand channel axis), p=0.7, s in [0.55,1.1]
    sel = torch.rand(B, device=z.device) < 0.15
    if sel.any():
        s = torch.ones(B, device=z.device); s[sel] = torch.empty(int(sel.sum()), device=z.device).uniform_(0.85, 1.15)
        xs = torch.linspace(-1, 1, 128, device=z.device)
        gx = xs[None, :] / s[:, None]                                   # sample positions in input
        grid = torch.stack([gx[:, None, :].expand(B, 96, 128), torch.linspace(-1, 1, 96, device=z.device)[None, :, None].expand(B, 96, 128)], -1)
        zz = Fn.grid_sample(z.reshape(B, 1, 96, 128), grid, mode='bilinear', padding_mode='zeros', align_corners=True).reshape(B, 6, 16, 128)
        mm = Fn.grid_sample(m.reshape(B, 1, 96, 128).float(), grid, mode='nearest', padding_mode='zeros', align_corners=True).reshape(B, 6, 16, 128)
        outside = (gx.abs() > 1)[:, None, None, :].expand(B, 6, 16, 128)
        # bilinear interpolation shrinks noise std -> renormalise per panel, fill outside with fresh noise
        zz = torch.where(outside, torch.randn_like(zz), zz)
        mm = torch.where(outside, torch.ones_like(mm), mm)
        sd = zz.flatten(2).std(2).clamp_min(0.3)[..., None, None]
        zz = zz / torch.where(sd < 1, sd, torch.ones_like(sd))
        z = torch.where(sel[:, None, None, None], zz, z); m = torch.where(sel[:, None, None, None], mm, m.float())
    m = m.float()
    # frequency shift (roll) and flip
    sh = int(np.random.randint(-30, 31)); z = torch.roll(z, sh, -1); m = torch.roll(m, sh, -1)
    fl = torch.rand(B, device=z.device) < 0.5
    z = torch.where(fl[:, None, None, None], z.flip(-1), z); m = torch.where(fl[:, None, None, None], m.flip(-1), m)
    return z, m

def up(C):
    return Fn.interpolate(C, scale_factor=UP, mode='nearest') if UP > 1 else C

def canvas(z, m, gap):
    """(B,6,16,128) -> (B,4,128,128) global-time canvas"""
    B = z.shape[0]; H = 128
    zr = z - z.median(-1, keepdim=True).values
    C = torch.zeros(B, 4, H, 128, device=z.device)
    p = torch.arange(6, device=z.device); r = torch.arange(16, device=z.device)
    t = p[None, :, None] * (16 + gap[:, None, None]) + r[None, None, :]          # (B,6,16)
    t = t + ((H - (6 * 16 + 5 * gap)) // 2)[:, None, None]                       # centre vertically
    bi = torch.arange(B, device=z.device)[:, None, None].expand(B, 6, 16)
    sign = torch.tensor([1., -1, 1, -1, 1, -1], device=z.device)[None, :, None].expand(B, 6, 16)
    C[bi, 0, t] = z * m; C[bi, 1, t] = zr * m
    C[bi, 2, t] = sign[..., None].expand(B, 6, 16, 128); C[bi, 3, t] = m
    return C


def inject(z, m, gap, yb, p_eti, p_trap):
    """Physics-based synthetic signals (labels follow the priority order):
    - ETI: a narrow drifting line on ONE straight track in global time, ON panels only -> label eti_candidate
    - trap: ON-only lines that do NOT connect (random offsets) or do not drift -> rfi_intermittent
      (only injected into noise / rfi_persistent cadences, whose priority is lower)"""
    B = z.shape[0]; d_ = z.device
    t = (torch.arange(6, device=d_)[None, :, None] * (16 + gap[:, None, None]) + torch.arange(16, device=d_)[None, None, :]).float()
    tc = t[:, [0, 2, 4]].mean((1, 2))[:, None, None]
    ch = torch.arange(128, device=d_).float()[None, None, None, :]
    on = torch.tensor([1., 0, 1, 0, 1, 0], device=d_)[None, :, None, None]
    r = torch.rand(B, device=d_)
    e = (r < p_eti) & (yb != 0)
    tr_ = (r >= p_eti) & (r < p_eti + p_trap) & ((yb == 7) | (yb == 8))
    if e.any() or tr_.any():
        drift = torch.empty(B, device=d_).uniform_(0.15, 1.3) * (torch.randint(0, 2, (B,), device=d_) * 2 - 1)
        c0 = torch.empty(B, device=d_).uniform_(20, 108)
        amp = torch.empty(B, device=d_).uniform_(0.6, 3.0); wid = torch.empty(B, device=d_).uniform_(0.5, 1.2)
        centre = c0[:, None, None] + drift[:, None, None] * (t - tc)                      # (B,6,16) connected track
        kind = torch.randint(0, 2, (B,), device=d_)                                       # trap type
        off = torch.empty(B, 6, 1, device=d_).uniform_(-40, 40)                           # disconnect offsets
        tr_centre = torch.where(kind[:, None, None] == 0, c0[:, None, None] + off + drift[:, None, None] * (t - t[:, :, :1]),
                                c0[:, None, None] + 0 * t)                                # non-connecting / non-drifting
        cen = torch.where(e[:, None, None], centre, tr_centre)
        line = amp[:, None, None, None] * torch.exp(-0.5 * ((ch - cen[..., None]) / wid[:, None, None, None]) ** 2) * on
        sel = (e | tr_)[:, None, None, None].float()
        z = z + sel * line * m
        yb = torch.where(e, torch.zeros_like(yb), torch.where(tr_, torch.full_like(yb, 6), yb))
    return z, yb

# ---------------- model -----------------
class Block(nn.Module):
    def __init__(s, i, o, st):
        super().__init__()
        s.c1 = nn.Conv2d(i, o, 3, st, 1, bias=False); s.b1 = nn.BatchNorm2d(o)
        s.c2 = nn.Conv2d(o, o, 3, 1, 1, bias=False); s.b2 = nn.BatchNorm2d(o)
        s.sk = nn.Sequential(nn.Conv2d(i, o, 1, st, bias=False), nn.BatchNorm2d(o)) if (i != o or st != 1) else nn.Identity()
    def forward(s, x): return torch.relu(s.b2(s.c2(torch.relu(s.b1(s.c1(x))))) + s.sk(x))

class ResNet(nn.Module):
    def __init__(s, w=40):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv2d(4, w, (5, 3), 1, (2, 1), bias=False), nn.BatchNorm2d(w), nn.ReLU())
        s.body = nn.Sequential(Block(w, 2 * w, 2), Block(2 * w, 2 * w, 1),
                               Block(2 * w, 4 * w, 2), Block(4 * w, 4 * w, 1), Block(4 * w, 6 * w, 2), Block(6 * w, 6 * w, 1),
                               Block(6 * w, 8 * w, 2))
        s.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(16 * w, 9))
    def forward(s, x):
        h = s.body(s.stem(x)); return s.head(torch.cat([h.mean((2, 3)), h.amax((2, 3))], 1))

def make(arch):
    if arch == 'res': return ResNet()
    import timm
    name = {'timm_resnet18': 'resnet18', 'timm_effb0': 'efficientnet_b0', 'timm_effb2': 'efficientnet_b2', 'timm_convnext': 'convnext_nano', 'timm_effb3': 'efficientnet_b3'}[arch]
    try: return timm.create_model(name, pretrained=True, in_chans=4, num_classes=9)
    except Exception as e: print('pretrained download failed, random init:', e); return timm.create_model(name, pretrained=False, in_chans=4, num_classes=9)

def predict(net, Z, M, G, tta=True):
    net.eval(); out = []
    with torch.no_grad(), torch.autocast(dev, enabled=dev == 'cuda'):
        for j in range(0, len(Z), 512):
            m = torch.from_numpy(M[j:j + 512]).to(dev).float()
            z = torch.from_numpy(Z[j:j + 512]).to(dev).float()
            if Z.dtype == np.uint8: z = norm_t(z, m)
            g = torch.from_numpy(G[j:j + 512]).to(dev)
            p = torch.softmax(net(up(canvas(z, m, g))).float(), 1)
            if tta: p = 0.5 * (p + torch.softmax(net(up(canvas(z.flip(-1), m.flip(-1), g))).float(), 1))
            out.append(p.cpu().numpy())
    return np.concatenate(out)

def pks_like(Z, M, seed=0):
    """deterministic 'PKS-like' version of held-out GBT data for shift-aware validation"""
    torch.manual_seed(seed); out_z, out_m = [], []
    for j in range(0, len(Z), 512):
        x = torch.from_numpy(Z[j:j + 512]).to(dev).float(); m = torch.from_numpy(M[j:j + 512]).to(dev).float(); B = len(x)
        x, m = byte_aug(x, m, 12., 12., 1.0); z = norm_t(x, m)
        xs = torch.linspace(-1, 1, 128, device=dev) / 1.0
        grid = torch.stack([xs[None, None, :].expand(B, 96, 128), torch.linspace(-1, 1, 96, device=dev)[None, :, None].expand(B, 96, 128)], -1)
        zz = Fn.grid_sample(z.reshape(B, 1, 96, 128), grid, align_corners=True).reshape(B, 6, 16, 128)
        out = (xs.abs() > 1)[None, None, None, :].expand_as(zz)
        zz = torch.where(out, torch.randn_like(zz), zz); sd = zz.flatten(2).std(2).clamp_min(0.3)[..., None, None]
        zz = zz / torch.where(sd < 1, sd, torch.ones_like(sd))
        mm = torch.where(out, torch.ones_like(m), Fn.grid_sample(m.reshape(B, 1, 96, 128), grid, mode='nearest', align_corners=True).reshape(B, 6, 16, 128))
        out_z.append(zz.cpu().numpy().astype(np.float16)); out_m.append(mm.cpu().numpy() > 0.5)
    return np.concatenate(out_z), np.concatenate(out_m)

def fit(Z, M, G, y, w_cls, epochs, seed, arch, evals):
    torch.manual_seed(seed); np.random.seed(seed)
    net = make(arch).to(dev); opt = torch.optim.AdamW(net.parameters(), 2e-3, weight_decay=0.02)
    bs = int(os.environ.get('BS', 128)); n = len(y); steps = epochs * (n // bs)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps, pct_start=0.15)
    lossf = nn.CrossEntropyLoss(weight=torch.tensor(w_cls, device=dev, dtype=torch.float32), label_smoothing=0.05)
    scaler = torch.amp.GradScaler(enabled=dev == 'cuda'); Y = torch.from_numpy(y)
    for ep in range(epochs):
        net.train(); perm = np.random.permutation(n); t0 = time.time(); L = 0
        for j in range(0, n - bs + 1, bs):
            idx = perm[j:j + bs]
            x = torch.from_numpy(Z[idx]).to(dev).float(); m = torch.from_numpy(M[idx]).to(dev).float()
            g = torch.from_numpy(G[idx]).to(dev); yb = Y[idx].to(dev)
            x, m = byte_aug(x, m); z = norm_t(x, m)
            if P_ETI + P_TRAP > 0: z, yb = inject(z, m, g, yb, P_ETI, P_TRAP)
            z, m = augment(z, m, True)
            with torch.autocast(dev, enabled=dev == 'cuda'):
                loss = lossf(net(up(canvas(z, m, g))), yb)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); L += loss.item() * bs
        if ep % 5 == 4 or ep == epochs - 1: print(f'   ep{ep} loss {L / n:.3f} {time.time() - t0:.1f}s', flush=True)
    return [predict(net, *e) for e in evals]

if __name__ == '__main__':
    D, EP, SEEDS = sys.argv[1], int(sys.argv[2]), int(sys.argv[3])
    arch = sys.argv[4] if len(sys.argv) > 4 else 'res'; PL = 'PL' in sys.argv
    tag = os.environ.get('TAG', arch); SMOKE = int(os.environ.get('SMOKE', 0)); os.makedirs('out', exist_ok=True)
    Z, M, G, meta = load(D, 'train'); Zt, Mt, Gt, tmeta = load(D, 'test')
    y = meta.label.map({l: i for i, l in enumerate(LABELS)}).values
    if SMOKE: Z, M, G, y = Z[:SMOKE], M[:SMOKE], G[:SMOKE], y[:SMOKE]; Zt, Mt, Gt = Zt[:SMOKE], Mt[:SMOKE], Gt[:SMOKE]
    cnt = np.bincount(y, minlength=9); w_cls = (cnt.sum() / (9 * np.maximum(cnt, 1))) ** 0.5; w_cls = w_cls / w_cls.mean()
    print('device', dev, 'arch', arch, 'train', Z.shape, 'class weights', w_cls.round(2), flush=True)
    if PL:
        pl = np.load('out/pl.npz'); sel, ypl = pl['sel'], pl['y']
        print('pseudo-labelled test cadences', sel.sum(), flush=True)
    oof = np.zeros((len(y), 8 + 1)); oof_pks = np.zeros_like(oof); pt = np.zeros((len(Zt), 9))
    for s in range(SEEDS):
        for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(Z, y)):
            Za, Ma, Ga, ya = Z[a], M[a], G[a], y[a]
            if PL: Za, Ma, Ga, ya = np.concatenate([Za, Zt[sel]]), np.concatenate([Ma, Mt[sel]]), np.r_[Ga, Gt[sel]], np.r_[ya, ypl]
            Zp, Mp = pks_like(Z[b], M[b], seed=k)
            pb, pp, pte = fit(Za, Ma, Ga, ya, w_cls, EP, 100 * s + k + int(os.environ.get('SEED_OFF', 0)), arch, [(Z[b], M[b], G[b]), (Zp, Mp, G[b]), (Zt, Mt, Gt)])
            oof[b] += pb / SEEDS; oof_pks[b] += pp / SEEDS; pt += pte / (5 * SEEDS)
            print(f'seed {s} fold {k}: F1 {f1_score(y[b], pb.argmax(1), average="macro"):.4f} | PKS-like F1 {f1_score(y[b], pp.argmax(1), average="macro"):.4f}', flush=True)
            np.save(f'out/oof_{tag}_partial.npy', oof); np.save(f'out/test_{tag}_partial.npy', pt); np.save(f'out/folds_done_{tag}.npy', np.array([s, k]))
        f = SEEDS / (s + 1)
        print(f'== after seed {s}: OOF F1 {f1_score(y, (oof * f).argmax(1), average="macro"):.4f} | PKS-like {f1_score(y, (oof_pks * f).argmax(1), average="macro"):.4f}', flush=True)
        print('per class (PKS-like):', dict(zip(LABELS, f1_score(y, (oof_pks * f).argmax(1), average=None).round(3))), flush=True)
        np.save(f'out/oof_{tag}.npy', oof * f); np.save(f'out/oofpks_{tag}.npy', oof_pks * f); np.save(f'out/test_{tag}.npy', pt * f)
    print('DONE', flush=True)


In [ ]:
import base64, os
os.makedirs('/kaggle/working/out', exist_ok=True)
open('/kaggle/working/out/pl.npz','wb').write(base64.b64decode('UEsDBC0AAAAIAAAAIQDwmnVb//////////8HABQAc2VsLm5weQEAEACQJwAAAAAAAPMGAAAAAAAAnZQ9ciRFEEYzXU4hbyBCButyADwIHIy1CMGKwCBYQtrAgT0FF0bSdL7vZUsWgtVoqqsrv9/69/sfv/vhfddf9fflw/3jLw+Xb24u//z87nJ7c/n148Onh7s/fvr48OH+ef3bu98f75/WH3+7+/P+6fuX775++rn96vbm883//Pmiqvv5Xz9/Xn/N/3U8e37KnusKj15Wrysvqy/bj8Xrn8eu47EWXt7N6zl1YPTxygzVIZzptetXlgR1+B0TBqCHXbcA7Pja85Gh7KzNESAoJ2ELWMPleA3dB1sUzG7BgcVsq8EybCOCTtX2nuOgx2f8OrCNpSONoAqCw/JGnNDzeBtiDVO+yBqSMqmJHRi0MjSDCQ0JIBAOarRagfR6KPQaeAyVHnNyYlRBskKfV0Y3Rs5XTyvoldfnZdAngGG3XxeTCNLZsiDK4M6RSQ0cyIJqtNPktLsnNgulolG6eYrsmZKLwxXVCzCupw+9SSSZ7v+qEKZrMAzou9KXUETyeISrE02VL70yktLola18t0nSRImkyYN+Z9v6u6u6AF5hJqc5MsCpJ91So9eonCqQeSI79E4Hk6K11Xcztat5V7/SqGL4YB0Ber2prFV+GvDz1worTidd8SyxeZUrts0EwDst4RNKbDmHW5hGzsleJgLSt4yvCkdbLNKsSAQo3SztAydWvqkSjDotprW+IOrEAVl1YQzVVb62EENavzWUer2VZrtTmaIt8wa7FD0cXi0Mb1jH/Ly1OFiZBBaqp6LYzjFLLNv8ZlVpD3iF11onJ5PqWBK4qQYTcHOOTYHAvjtiF/FH2zMchV3JBUeNlXtKVvuheqJrIMcTJ3F1Y3VIiYXMXgjUi4VONiUkyK0yih2KS5GVRWmjaBlXhFtBSGydx2kGb8wVID5KDE6vMOzDeWJdQ5AcJVnVOhzOQyXgpZGyHSvXXTKqETY1gguByk2xVhaIad6PCejaLURp5e6VD+TIQCwsPqWog/xkXS6LU1jTdV0Xu72pFENoag7g8GZ76OzkRIZ+83GFrBZSUFm/Xg0kxYRM7YCr5dJjdMAEpVSmJeVJWEJGbUpxggwHlCkooOlAGZ5el6lGEJmwBct6ry49SV/CzcERmHAuEgu0W4irVhUR31CSuOpemOd7+hJ0zCmWa34mEE25EhHMjvC0MeU6tTzwloPKjjLKpKWBy0z+loCu3vyVUAy1ghOVNicxXpqdODs4OjreOgfqWuyLubyeUWW87WE2yEjaIvY6JfJlaun4PaorQoqRQmngnf+iE0eoIScvmzuFjUOLXAM/KZVFeINhc4RtcmCIgTpId5wHb0e40eJVveCgVwdjCS/pU1R0HZT5rKIkvrGHawH0rn8cltgqyalhTpluGqk6VtKMldjVqeivBCr6MyQTwwFTkmm+RFnCG4nnkCHfgRZd56heWefwdnTdhfEpPVQ/+PqqDwrjci5YwA0QS7lqUZRFZiOsOCmf4k3ca0JN+qOmbhoV2r45+EgihBmtoyZ2tDbJclWDdqZLIeTZdXLBRF3epyZYtYLK1Ek5Biilp2mybpSoUTMuk4GUgTgVUHSfTOlPm10bWzZloEhtbiUisTqFDPCkdUTpRYJnmwxdEiNN9CGkYmVMAWkw5E5Ad8KAIIG2ljmogwQsyu9iKqVc2WYNAnEjg3iyZIdZZdd6ksB3wO5YNXxWGWjajJYVciB9sSi+9dXx1jSCuY7Yz9IT2Ux/lbiz6u7umbkY0sbkIhhUXjdvGiTYPKna8PCdGMhWh7FEYGh0tqWt2dow3QtW28A9tQyWhKl+cnE1xoFQwif+c6IdQKKZHYEyPt46y4q2yrnLMuXSmwcbLoHENjkug6FOCxbgHeLRoYcOMqgmzoViNtonDhSvMziaICMvZqWETvgtgU01acxCn9QLLpgWnCjNB1NIEXgZtlmUcRBMmqRupJkVQAnb9ADynb3YzoPJhurrku2X5ppwOmOaWzzy5eJJQ6wN7yiqMmLfA6ImvHwghhoh0Ola4AZK7Q1EpATRTxN5nFrlWQwsWUqiwpyjK+UVRdJTIGud76Cdo5x/su0kDjkYV1v7O/j0qiiQoWlfMFZGgQxa7tFq6YpR+irndMiKX2IRulMG3qZaa9ioNrlgKieHE11NIxMFqRjkQVtilMg6/0IatRMgaM4zEIZH8KB5ctcA3j2UHZWvIYnVAbWV0zppJTirUHlQCzmTkYPwO5mjJwfufK6K9po14BS5+MsRbIifgqiaZ0V9T0REWz0nvJmVfi/lwBStd9EVqMgQjmqUnG8KWSKO0EG71EnjEwHaPrXshYQ4swWPhkX5vc5CjuA5FtjdVwxXt7qFslg8xVaGr+fNMkJX6gFnmNC5CiNVGiSHzv8BUEsDBC0AAAAIAAAAIQDPo1VW//////////8FABQAeS5ucHkBABAAsJQAAAAAAAALDQAAAAAAAJ3cu8pkxxmF4Z6z0VVM9lswkRHIGMXObJw4cCQGzW8kEJKYMUqEr0I3rDHWDrTg4a1WJ83u3oeq77jWqur++e///Ns//vXk9uPtp4d3jx++ev/wl9cPX3zz54c3rx/+/f37/7x/+92X379/9/i/z//69tsPjx8///D12x8ePx7/8bPPP/vTm0/fvP7v69/7+uTp7f+vZ7++/+HX95fz/mKOn875e7zX7/fX/Z7P9y/n8yfz/nS+3/ten1/32eufzfGLOdb9ns77jnPn/Wruq8/3fvv5jv/VfL/X7fzXv3vdfr7zeD7v+5zLfuuX9e/Gw8aV7L/P2/ErXl7M+9pTn1/Hl503Xneez+b9um79ovnv+K/77/jXL+X/tUfZv/x4vW/87nU7rj1/82rnp+P1k+rSXrfjXDtv/G9+KY93vpunGv/Ov/yuerfz3XzZ67ZOndZVxefaf/258bh2qfjdeN+6XefvOGSvrU/r7z1ee8lPilPl085P/XPjU/17n7Pz0Tj3WHVOfX7vv/Pa+Ku+vfml+Fk7bdyXX9aOVdeUr6rPO47LToqLjQ/VOx1vXbvNsebx/Len3x1nm68bF+tf9X/hmPJ35V3VT8XLzm/no/qjvN58Er5aHHIah6p7um7P23nv8+W34gXlD+H+HZfwq+rE9fn1UlytvQt/Cg+rfqhflv3K3sIzwinC6fu8qtc6X/V17ab6r/yRHZWv16v42+Zj8RH1p81bxbP43/q9eJfiUM/ZurV2EL5VvZVdNo+ED1V3hRuVj8JhG4+yr/Bl8b+9fp9bfEpxVnlfeEhxqfhRnZbuoPedn/JSx6rn4mUaR/Vv8crqZ6UHnfI91SfxG/mxeN5+r/6mvF/+Uzx0cV/hnOon1S+Lp+1zhD+Lh+79VD/3fltXN773/OJh1R8q3qSbqh8prq+X/K16Ir1tr188oTipuqG4Pa330ql2fuLppUOt/wovKg73/LWH6ob6p3QX6QCyy8aL8mTvs/Vieenm1Z4nPiG8fC/OFK4u/iO7C5+obl2vzZfS12s9Qfhi43n5UsWN+kXpLLK/dEn5Sf4/jT/hWPVL1efrdcpnd/6qN+IHWm865d/F1+VPrfuoj0m3UL2W/TT+5WNrJ9Wbza/SAaR/C08qzvd+4hGF8zU+4R71QdV59VHpuPJr4aI9X/y9dEHV/41L4WTFnXRaxfs+f+NT81ec1TqieN4pnqpxC2eLV1T/lx33euGstee9/FTxWnVv64k+l35/ysOul/h69c+dZ+nA0udkV8WD4kr1qfhLfS88tHlf67yar+yj99INC/+s/8uvGv/GjfjzXrd1rnRvHWsfwcbJ1mHlxykuqH5wih8Lr6jvLT5ce8huwnXSWcV/K3+2Pm2cKI7vxZW1T0P32bpedVT5XX2leIN0V/WjrTeyz/pF+9V0Xek2itOdp3iWcITwvK5bXHRaX4snKi/XfrUPpfqm7l/9tfq0cGzhaeFyPV88V/bWfIVHlc/q71XfpT9U/ZaecrpOI9wuPUX5I5ygPis8W/hCdUy8qeqo6t+pTrb30/3F76tubZ2svqP6XfGo/QSn9VX7igqPiIcLP6hf6Dna53raj2sdRfF5Ol/lt/Tt4u3C4zveU32p8ITWj6qOCedI1934UF0qPl+8qvZD1jqpeJ3wSuHu6jeqx2uv6gc7762nZefyz9Z52UW4QX1e/WrtIb9J31D8lQ6rOq/8KV1T+pnsqXwuXXLHsbhXfpGfxVNkh8KRFReFA5QvpftovUfrDlrPk95xqh8K/2z+Kq7ET5S/4sV1H623S3+WrlE8UHEi+0m3Kx1C55Uuqnxa+23+7jylu6svLQ6rdRHhhcKzxcOFy4R/tS6l9UnhIa3Lil8U35Rer7jd88WrhY/lH+GR6qeFd0rnkm6juK46W7hL+H/ru/Bl+b10QOWp4lL1c8/bPlJ1v/Sdimv5eecvHCWeIHyr74VDpMNU3RK+2XntfOp3QcqT9Vf18dKZlN/St4WrC/dLb6+6Vr/TEg4o3iv+pLpW8xfuEX4TH6vxLh+quitcqHyueicdqfTLfZd+tMenvFJ1fcdV+1rK7sorzVN15rTeiT/Iv1t/VUdk9/WL9puc1udTXVj2qGPZS/hT/El4SXYqfVL9Tvxhr7/N99V3t47Wup340l4n3lX6sepI6Q6nutbpfsbCMeIRp/VQeqF4lPCD6pXibv1w7zqK1qG0/7j029IdFgfI38U7VafUNxQ/8pu+135V6XSn/hR/rLyR3YtPl36jOqF6qf2aVRdKZxVOVTwXjt37yi/qL6d4de1V+xnFX0tn3H4vvUt22by8zbHqivL63vze9TPx58Jf6kPaN1XrNTv/0m9q/6V4kOyn+NB4xV9VR8SLtc5X9UL+2XlV/S3cJD1M/hW+qXyq+JL9tE4n/iEeVHqP8LbyVbhR/Vt2UZ9Snit/hacKX5fOqXir/b9a75e9xJ+KJwvn6v47Puly4quFO9Y+4lna56T8uF76HWv97kA8VuuMmo/yWPxKcSz8ojzd8/V7LeHYva94hPiZ9CzVTelSssdtPlfcF4+STqk4ER68XsI1pa+vXaufC5dLD6j/tyq9as9Tv9d6gO5buoX05Y1r2U/rfuJVsnPposofPWftIR1N/lefXHsU/hBv1HpIrQ/IPqWLCpcqntVX1o/V52rfrfrB+m2vk66xcav+UP28eITwutanFF/rl51f6bjKz8J9p7im6qR4WOGhHb/ydXGo8qh0kcqb4subD8LfwnX7vXQm1c+an3Sw0pUKX1RfFm/TvqBaR9h40rqgcGrx2MJ/mk/pcdJltH+rdPbK/7Wnfpep/qA6r3qq80q/K31IfKj6nXBdvRfPrvitfYnSjU75Z+mapUcWjilduvTx0tuF6079pH0lhQOv4+u1ebTzka5fOnv1AdlR9UXXFY9a3Vn8WjqP4u5UTz7V6+r3gVqnUp1VXd88Uz+QzrJ2vMZzvcRXb/O56ohwgfRe4d3N69t8Xrin4mDtuefv88RjpAuoj1efKj1cumitcypelN/yj/pF3Uc4rnQ/5bfqWq2zV30vvlN9WX5X/pbOW3hNvK/6f+Vj6XDSbcqPwuvSzcVXdv7a56N6oDiQ3/Z3IsWHlBfiPYrH0inEt7T+I/uVjiB9SXlV9hHvVf/b+FvepHohP+44djzXq+rG9tm9j3RE9S+tLyxf3PO3vwkvnfKf8q/Wj5T3wrvXq3Dr+rX8KBxb+Gr9q7gVbhEvqf4rP+3/dAhvFc6Wvi4dueplrSOJT2hc4n/6fYF0gOLp8v+p34oHKJ6ED2t9vPjRjrdw1SkOPMUz1c9UN3Se6lPxf+lvVRcVx4Xni3feO07pb1v/b/O56scpLyrcp363/tD+lZp36W3SxfU/mcpXrWNtfCz/23FKR1V8SsfRPJSftQ5QupT0cMVp6WX6XbT0D9UvxWntR6jn6f6LL/Y+pzqbePe9OEs4Q7hLfhc+0TqmnqP1PfUb8cj1u/hZrcPU7+/u7S/1PK0DFp8vHb50neLZld/r79t8L57y6ren5/pZ4STVvz1P+r78LHyrulj1Tutz9T+GpRtWHa1+Xf3g3vWo+p+R+l+hjbftW2tf5U/hgYpv6Z/1u4/SZwoHFo7TfsLC0+rbtR6o9dDCA6rTG7f6vf/mnT6XXeq520/Fo2o/lPxc+rri/LQ/1v4H4d36vZPwssZX+/GFA4SXiudKh9E8t37sPITrCx/VPopaF5COoXkJn6uu7Dpbxekpn5KuLB6rvqi+oviv/+fRusjWZ62jqO8rDqQDixee9uvCfZWXyyOkv+089D90hSulJ9R6dfFi8V/VMdVV6X+Fj4VLSkfa8er/jlVPpXesnRRfxeNLrzvFaTt/6VzFk4TrpQtr3VnPq9/jC/8Xntf/GW383OZYeFz9Q3VR41aeiF+s3q/6V/9joLwTv15/S+eQ3iM8tXF1+nsZ8R7Vu7Wj4qf2+Z/WgdLlFM+6r3CK+q76lnRH2e16La6X/ly6ktY7ilftc5RX6kPSKdRnVE9qP+rmw+lz5afSx/a50if0vzWlP5WuV7y/9r0Vrqy+pH56un6hfNP4pMsUn7rNeapv1bfXvtIpVYdO9zeUbnfqj9JtijcX3lOd0HyF46UvSLfQ85SPque67nqV/XT96bH0plM9WnVCeHDvL1wq3Km+urhE8VT5W/W6+HjVk6qftS4mvXb/v6H0Js1X9hHuLZ6sfi3/lH5dPLr6aI2v/F6/Txa/uF7Cu7qP8qRw/dpXfVh6xSkuFy4Qzy6cvn7Xeof6uf5nRevy+v8P/d+S9DfFo54r/an0o7Xj6f911O8kdN/Su9QfxNuLB0knFa+UfiD8vM9f/0inKv+qHm7dkA6hOlz1cHlfjVP5pvwTTpVOprpZ9Vm6ofQszUfxvLhK9t3xnu47vl6lp9f+nR2v+k3t/z/l2dKfdF/9v4N0B+EC1dVfAFBLAQItAy0AAAAIAAAAIQDwmnVb8wYAAJAnAAAHAAAAAAAAAAAAAACAAQAAAABzZWwubnB5UEsBAi0DLQAAAAgAAAAhAM+jVVYLDQAAsJQAAAUAAAAAAAAAAAAAAIABLAcAAHkubnB5UEsFBgAAAAACAAIAaAAAAG4UAAAAAA=='))
import numpy as np; z=np.load('/kaggle/working/out/pl.npz'); print('pseudo-labels', z['sel'].sum())

In [ ]:
import glob, os, subprocess, time
os.chdir('/kaggle/working')
D = os.path.dirname(glob.glob('/kaggle/input/**/train_cadences.npy', recursive=True)[0]); print('DATA', D)
!pip -q install timm 2>/dev/null
EPOCHS = 30
p0 = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=0 SEED_OFF=16000 TAG=effb2_fixs2 P_ETI=0.05 P_TRAP=0.04 python -u cnn_r2.py {D} {EPOCHS} 1 timm_effb2 PL > out/log_effb2_fixs2.txt 2>&1', shell=True)
p1 = subprocess.Popen(f'CUDA_VISIBLE_DEVICES=1 SEED_OFF=16500 TAG=effb0_fixs2 P_ETI=0.05 P_TRAP=0.04 python -u cnn_r2.py {D} {EPOCHS} 1 timm_effb0 PL > out/log_effb0_fixs2.txt 2>&1', shell=True)
print('started')

In [ ]:
while p0.poll() is None or p1.poll() is None:
    for f in ['out/log_effb2_fixs2.txt', 'out/log_effb0_fixs2.txt']:
        lines = [l for l in open(f).read().splitlines() if 'fold' in l or '==' in l or 'Error' in l or 'per class' in l or 'pseudo' in l]
        print(f, '|', *lines[-3:], sep='\n  ')
    print('-' * 60, time.strftime('%H:%M:%S'), flush=True); time.sleep(300)
print(open('out/log_effb2_fixs2.txt').read()[-3000:]); print(open('out/log_effb0_fixs2.txt').read()[-3000:])

In [ ]:
import zipfile
from IPython.display import FileLink
with zipfile.ZipFile('r2_outputs_run10.zip', 'w') as z:
    for f in glob.glob('out/*.npy') + glob.glob('out/*.txt'): z.write(f)
FileLink('r2_outputs_run10.zip')